# NeRF (Nerfacto) — bear-mask

Trains Nerfstudio's **nerfacto** on the masked bear capture and saves the model, rendered
comparisons and training metrics to Google Drive.

**Before running**
1. *Runtime → Change runtime type →* a **GPU** (T4 works; L4/A100 is faster).
2. Upload `data/nerfstudio/bear-mask.zip` from the repo to **`MyDrive/3d-reconstruction/bear-mask.zip`**
   (upload the zip as-is, don't unzip it). It was produced locally by `scripts/prepare_nerfstudio.py` from the
   undistorted COLMAP model (`colmap/bear-mask/dense/0`): 87 images at 1600×1062, stored as RGBA PNGs whose
   alpha is the rembg mask warped into the undistorted camera, plus `transforms.json` and `sparse_pc.ply`.
3. Run the cells top to bottom. **Section 2 restarts the Python kernel once after installing** (this is
   intended). Then continue from section 3. Every shell command is echoed
   and appended to a log on Drive, and each stage checks its outputs before moving on.

**Settings shared with the other notebook (keep them identical for a fair comparison)**
`--pipeline.model.background-color white` (the alpha matte is composited onto white),
`nerfstudio-data --downscale-factor 1 --eval-mode interval --eval-interval 8`
(every 8th image is held out for evaluation, 11 of 87).

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Install Nerfstudio (restarts the kernel)

* `gsplat==1.4.0` is pinned by `nerfstudio==1.1.5`. Its prebuilt wheels only cover Python 3.10, and Colab
  runs a newer Python, so it is installed with `BUILD_NO_CUDA=1`. Its CUDA kernels are then JIT-compiled the first
  time they are used. Nerfacto never uses them.
* Pip will warn about conflicts with Colab's preinstalled packages (protobuf, opencv, tensorflow). These are expected.
* Nerfstudio downgrades protobuf and opencv, so the cell restarts the Python kernel after installing to make sure
  nothing stale stays loaded (installed packages are kept). **Continue from section 3.** Re-running this cell later
  (or after a restart) skips the install.

In [ ]:
import importlib, importlib.util, sys
print(sys.version)

if importlib.util.find_spec('nerfstudio') is None:
    !pip install -q ninja
    !BUILD_NO_CUDA=1 pip install -q gsplat==1.4.0
    !pip install -q nerfstudio==1.1.5
    importlib.invalidate_caches()   # let this process see the newly installed packages
    assert importlib.util.find_spec('nerfstudio') is not None, 'nerfstudio install failed — see the pip output above'
    print('Installed. Restarting the Python kernel now; when it is back, continue from section 3.')
    get_ipython().kernel.do_shutdown(restart=True)   # clean kernel restart; the VM and installs are kept
else:
    print('nerfstudio already installed — skipping. Continue from section 3.')

## 3. Mount Drive and set paths
Run this after every runtime restart. Outputs go to Drive so they survive a disconnect.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/3d-reconstruction'
DATA_ZIP = f'{DRIVE_ROOT}/bear-mask.zip'
DATA = '/content/data/bear-mask'                # unzipped to fast local disk
TEST_OUT = '/content/test_runs'                 # smoke test, not kept
OUT = f'{DRIVE_ROOT}/results/nerfacto'          # full run, kept on Drive
LOG_DIR = f'{OUT}/logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Nerfstudio 1.1.5 calls torch.load() without weights_only; torch>=2.6 defaults it to True,
# which breaks resuming, ns-eval, ns-render and ns-export.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
assert os.path.exists(DATA_ZIP), f'Upload bear-mask.zip to {DATA_ZIP}'

### 3b. (Optional, recommended for the full run) tiny-cuda-nn
Without it, Nerfacto falls back to a pure-PyTorch hash grid that is several times slower. In local testing on
an RTX 3070 that meant ~0.3 s/iter, or roughly 2.5 h for 30k iterations. Building it takes ~10–20 min (once per
runtime; skipped if already built). Nerfacto picks it up automatically if the import below succeeds. If the
build fails, skip this cell; training still works.

In [ ]:
import importlib.util, torch
if importlib.util.find_spec('tinycudann') is None:
    major, minor = torch.cuda.get_device_capability()
    os.environ['TCNN_CUDA_ARCHITECTURES'] = f'{major}{minor}'
    !pip install -q --no-build-isolation "git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch"
!python -c "import tinycudann; print('tiny-cuda-nn OK')"

## 4. Verify the environment

In [ ]:
import torch, nerfstudio, gsplat
from importlib.metadata import version
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '|', torch.cuda.get_device_name(0))
print('nerfstudio', version('nerfstudio'), '| gsplat', gsplat.__version__)
assert torch.cuda.is_available()

## 5. Unpack and check the dataset

In [ ]:
import json, zipfile
from PIL import Image

if not os.path.exists(f'{DATA}/transforms.json'):
    zipfile.ZipFile(DATA_ZIP).extractall('/content/data')

meta = json.load(open(f'{DATA}/transforms.json'))
images = sorted(os.listdir(f'{DATA}/images'))
first = Image.open(f"{DATA}/{meta['frames'][0]['file_path']}")
print(len(meta['frames']), 'frames |', len(images), 'images |', meta['w'], 'x', meta['h'], '|', first.mode)
assert len(meta['frames']) == len(images) == 87
assert first.mode == 'RGBA' and first.size == (meta['w'], meta['h'])
assert meta.get('ply_file_path') == 'sparse_pc.ply' and os.path.exists(f'{DATA}/sparse_pc.ply')
first

## 6. Helpers
`sh()` streams a command's output, appends it to a log file and raises if it fails.

In [ ]:
import glob, subprocess, sys

def sh(cmd, log):
    print('$', cmd, flush=True)
    with open(log, 'a') as f, subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        f.write(f'$ {cmd}\n')
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f'exit code {p.returncode} — see {log}')

def train_cmd(out_dir, timestamp, iters, resume_dir=None):
    # NB: ns-train runs `--max-num-iterations` steps *after* the loaded checkpoint, not up to it.
    cmd = (f'ns-train nerfacto --data {DATA} --output-dir {out_dir} --experiment-name bear-mask '
           f'--timestamp {timestamp} --vis tensorboard --max-num-iterations {iters} '
           f'--steps-per-save 2000 --pipeline.model.background-color white ')
    if resume_dir:
        cmd += f'--load-dir {resume_dir} '
    return cmd + 'nerfstudio-data --downscale-factor 1 --eval-mode interval --eval-interval 8'

def run_dir(out_dir, timestamp):
    return f'{out_dir}/bear-mask/nerfacto/{timestamp}'

def checkpoints(run):
    return sorted(glob.glob(f'{run}/nerfstudio_models/step-*.ckpt'))

def ckpt_step(path):
    return int(os.path.basename(path)[5:-5])   # step-000029999.ckpt -> 29999

## 7. Smoke test (500 iterations)
Checks the whole chain (training, checkpointing, evaluation) on a short run before the full job.

In [ ]:
TEST_LOG = f'{LOG_DIR}/test_train.log'
open(TEST_LOG, 'w').close()   # fresh log on every attempt
sh(train_cmd(TEST_OUT, 'test', 500), TEST_LOG)

test_run = run_dir(TEST_OUT, 'test')
assert checkpoints(test_run), 'no checkpoint written'
print('checkpoint:', checkpoints(test_run)[-1])

sh(f'ns-eval --load-config {test_run}/config.yml --output-path {test_run}/eval.json', f'{LOG_DIR}/test_eval.log')
print(json.load(open(f'{test_run}/eval.json'))['results'])

## 8. Full training (30,000 iterations)
Roughly 1–3 h depending on GPU and whether tiny-cuda-nn is installed.
Checkpoints are written to Drive every 2,000 steps. If the runtime disconnects, re-run sections 3–6 and
then this cell. It resumes from the latest checkpoint and trains only the remaining steps. Once training is
complete, re-running this cell does nothing.

In [ ]:
ITERS = 30000
RUN = run_dir(OUT, 'full')
done = checkpoints(RUN)
start = ckpt_step(done[-1]) + 1 if done else 0

if start >= ITERS:
    print('already trained to step', start - 1)
else:
    print(f'resuming from step {start}' if done else 'training from scratch')
    sh(train_cmd(OUT, 'full', ITERS - start, f'{RUN}/nerfstudio_models' if done else None),
       f'{LOG_DIR}/full_train.log')

last = checkpoints(RUN)[-1]
print('final checkpoint:', last)
assert ckpt_step(last) == ITERS - 1, 'training stopped before the last step'

## 9. Evaluate on the held-out images
`ns-eval` reports PSNR/SSIM/LPIPS averaged over the 11 eval views. For each view it saves a
side-by-side **ground truth | render** image (`eval_img_XXXX.png`).

In [ ]:
EVAL_DIR = f'{RUN}/eval'
sh(f'ns-eval --load-config {RUN}/config.yml --output-path {EVAL_DIR}/metrics.json '
   f'--render-output-path {EVAL_DIR}/comparisons', f'{LOG_DIR}/eval.log')

results = json.load(open(f'{EVAL_DIR}/metrics.json'))['results']
for k, v in results.items():
    print(f'{k:>16}: {v:.4f}')
comparisons = sorted(glob.glob(f'{EVAL_DIR}/comparisons/eval_img_*.png'))
assert len(comparisons) == 11, len(comparisons)

In [ ]:
from IPython.display import display
for path in comparisons[:4]:
    print(os.path.basename(path), '— left: ground truth, right: render')
    im = Image.open(path)
    im.thumbnail((1400, 1400))
    display(im)

## 10. Training curves
Exports every TensorBoard scalar to CSV and plots the loss and PSNR curves. If training was resumed, steps
logged after the last checkpoint of the interrupted run are replaced by the resumed run's values.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(RUN, size_guidance={'scalars': 0})
ea.Reload()
rows = [(tag, e.wall_time, e.step, e.value) for tag in ea.Tags()['scalars'] for e in ea.Scalars(tag)]
scalars = (pd.DataFrame(rows, columns=['tag', 'wall_time', 'step', 'value'])
           .sort_values('wall_time').drop_duplicates(['tag', 'step'], keep='last')
           .sort_values(['tag', 'step']).drop(columns='wall_time'))
os.makedirs(f'{RUN}/metrics', exist_ok=True)
scalars.to_csv(f'{RUN}/metrics/training_scalars.csv', index=False)
print(sorted(scalars.tag.unique()))

tags = [t for t in scalars.tag.unique() if any(s in t.lower() for s in ('loss', 'psnr'))]
assert tags, 'no loss/psnr scalars found'
fig, axes = plt.subplots(len(tags), 1, figsize=(8, 2.6 * len(tags)), sharex=True, squeeze=False)
axes = axes[:, 0]
for ax, tag in zip(axes, tags):
    d = scalars[scalars.tag == tag]
    ax.plot(d.step, d.value, lw=1)
    ax.set_title(tag, fontsize=10)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('step')
fig.tight_layout()
fig.savefig(f'{RUN}/metrics/training_curves.png', dpi=150)

## 11. Fly-through video
Interpolates between the training camera poses and renders at half resolution.

In [ ]:
VIDEO = f'{RUN}/renders/interpolate.mp4'
sh(f'ns-render interpolate --load-config {RUN}/config.yml --output-path {VIDEO} '
   f'--pose-source train --order-poses True --interpolation-steps 3 --frame-rate 24 --downscale-factor 2',
   f'{LOG_DIR}/render.log')
assert os.path.getsize(VIDEO) > 0

from IPython.display import Video
Video(VIDEO, embed=True, width=800)

## 12. Export a point cloud
Samples the trained field into a coloured point cloud (`point_cloud.ply`) that you can compare with COLMAP's
`fused.ply`. Nerfacto doesn't predict normals by default, so normals are estimated with Open3D.

In [ ]:
EXPORT = f'{RUN}/export'
sh(f'ns-export pointcloud --load-config {RUN}/config.yml --output-dir {EXPORT} '
   f'--num-points 1000000 --remove-outliers True --normal-method open3d', f'{LOG_DIR}/export.log')
assert os.path.exists(f'{EXPORT}/point_cloud.ply')

## 13. What was saved to Drive

In [ ]:
for dirpath, _, files in sorted(os.walk(OUT)):
    for name in sorted(files):
        p = os.path.join(dirpath, name)
        if 'comparisons' not in p:
            print(f'{os.path.getsize(p) / 1e6:9.1f} MB  {os.path.relpath(p, OUT)}')
print(len(comparisons), 'comparison images in', f'{EVAL_DIR}/comparisons')